## 0. Install

In [ ]:
# !pip install mediapipe-silicon opencv-python pandas scikit-learn

## 1. Import Dependencies

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic = mp.solutions.holistic
mp_pose = mp.solutions.pose

## 2. Make Some Detections

In [ ]:
cap = cv2.VideoCapture(0)
# Initate holistic model
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, image = cap.read()

        # Recolor Feed
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Make Detections
        results = pose.process(image)

        # Recolor image back to BGR for rendering
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark
        except:
            pass

        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                 mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))

        cv2.imshow('Raw Webcam Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)
cv2.waitKey(1)

In [ ]:
mp_drawing.plot_landmarks(
    results.pose_world_landmarks, mp_holistic.POSE_CONNECTIONS)

In [ ]:
pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.7, model_complexity=2)
def detectPose(image, pose, display=True):
    '''
    This function performs pose detection on an image.
    Args:
        image: The input image with a prominent person whose pose landmarks needs to be detected.
        pose: The pose setup function required to perform the pose detection.
        display: A boolean value that is if set to true the function displays the original input image, the resultant image,
                 and the pose landmarks in 3D plot and returns nothing.
    Returns:
        output_image: The input image with the detected pose landmarks drawn.
        landmarks: A list of detected landmarks converted into their original scale.
    '''
    # Copy example image
    output_image = image.copy()

    # Convert color image BGR TO RGB
    imageRGB = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Perform pose detection
    results = pose.process(imageRGB)

    # Get width & height of the input image
    height, width, _ = image.shape

    # Initialize an empty list to store detected landmarks
    landmarks = []

    # Check if landmarks are detected
    if results.pose_landmarks:

      # Draw landmarks
      mp_drawing.draw_landmarks(image=output_image, landmark_list=results.pose_landmarks, connections=mp_pose.POSE_CONNECTIONS)

      # Iterate through detected landmarks
      for landmark in results.pose_landmarks.landmark:

        # Add landmark to the list
        landmarks.append((int(landmark.x * width), int(landmark.y * height), (landmark.z * width)))

    # Compare original image with pose-detected image
    if display:

      # Draw original & output images
      plt.figure(figsize=[22,22])
      plt.subplot(121);plt.imshow(image[:,:,::-1]);plt.title("Original Image");plt.axis('off');
      plt.subplot(122);plt.imshow(output_image[:,:,::-1]);plt.title("Output Image");plt.axis('off');

      # Display 3D landmarks
      mp_drawing.plot_landmarks(results.pose_world_landmarks, mp_pose.POSE_CONNECTIONS)

    # Otherwise, return output_image and landmarks
    else:

      return output_image, landmarks
# Pose detection function start

image = cv2.imread('./benchpress.jpg')
detectPose(image, pose, display=True)

In [ ]:
image = cv2.imread('./bench press2.jpg')
detectPose(image, pose, display=True)

In [ ]:
image = cv2.imread('./bench press3.jpg')
detectPose(image, pose, display=True)

## 3. Save Video

In [ ]:
cap = cv2.VideoCapture(0)
if cap.isOpened:
    file_path = 'exercise.avi'
    fps = 30
    fourcc = cv2.VideoWriter_fourcc(*'DIVX')            # Encoding Format
    width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    size = (int(width), int (height))                   # Frame Size

    out = cv2.VideoWriter(file_path, fourcc, fps, size) # Create VideoWriter Object
    while True:
        ret, frame = cap.read()
        if ret:
            cv2.imshow('camera-recording', frame)
            out.write(frame)                            # Save this avi file
            if cv2.waitKey(int(1000/fps)) != -1:
                break
        elif cv2.waitKey(10) & 0xFF == ord('q'):
            break
        else:
            print('no file!')
            break
    out.release()                                       # Close
else:
    print("Can`t open camera!")
cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)
cv2.waitKey(1)

## 4. Capture Landmarks and Export CSV

#### 4_1. Import Dependencies

In [ ]:
import csv
import os
import numpy as np
import matplotlib.pyplot as plt

#### 4_2. Capture Landmarks

In [ ]:
landmarks = ['class']
for val in range(1, 33+1):
    landmarks += [f'x{val}', f'y{val}', f'z{val}', f'v{val}']

In [ ]:
landmarks[1:9]

In [ ]:
with open('coords.csv', mode='w', newline='') as f:
    csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
    csv_writer.writerow(landmarks)

In [ ]:
# def export_landmark(results, action):
#     try:
#         keypoints = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten()
#         keypoints.insert(0, action)

#         with open('coords.csv', mode='a', newline='') as f:
#             csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
#             csv_writer.writerow(keypoints)
#     except Exception as e:
#         pass

In [ ]:
def export_landmark(results, action):
    try:
        keypoints = [action] + [coord for res in results.pose_landmarks.landmark for coord in [res.x, res.y, res.z, res.visibility]]

        with open('coords.csv', mode='a', newline='') as f:
            csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
            csv_writer.writerow(keypoints)
    except Exception as e:
        pass

In [ ]:
print(len(landmarks))

In [ ]:
print(results.pose_landmarks)

In [ ]:
export_landmark(results, 'correct_bench')

In [ ]:
cap = cv2.VideoCapture('exercise.avi')
# Initate holistic model
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:

    while cap.isOpened():
        ret, image = cap.read()

        if not ret:  # 비디오 재생이 완료되면 루프를 종료합니다.
            break

        # Recolor Feed
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Make Detections
        results = pose.process(image)

        # Recolor image back to BGR for rendering
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                 mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))
        k = cv2.waitKey(1)
        if k == ord('c'): # c
            export_landmark(results, 'correct_bench')
            print('correct_bench')
        if k == ord('i'): # i
            export_landmark(results, 'incorrect_bench')
            print('incorrect_bench')

        cv2.imshow('Raw Webcam Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)
cv2.waitKey(1)

In [ ]:
for landmark in mp_pose.PoseLandmark:
    print(landmark)

In [ ]:
print('LEFT SHOULDER')
print(landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value])

print('LEFT ELBOW')
print(landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value])

print('LEFT WRIST')
print(landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value])

In [ ]:
# Angle calculation function
import math
def calculateAngle(a, b, c):
    a = np.array(a) # first
    b = np.array(b) # mid
    c = np.array(c) # end

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle

In [ ]:
shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]

In [ ]:
calculateAngle(shoulder, elbow, wrist)

In [ ]:
import cv2
import numpy as np
import math

cap = cv2.VideoCapture('squat_min.mp4')


while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        break

    # Recolor Feed
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame.flags.writeable = False

    # Make Detections
    results_pose = pose.process(frame)

    # Recolor image back to BGR for rendering
    frame.flags.writeable = True
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    mp_drawing = mp.solutions.drawing_utils
    mp_drawing.draw_landmarks(frame, results_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2))

    # 각도 계산을 위한 랜드마크 추출
    if results_pose.pose_landmarks is not None:
        landmarks = results_pose.pose_landmarks.landmark
        left_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP].x, landmarks[mp_pose.PoseLandmark.LEFT_HIP].y]
        left_knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE].y]
        left_ankle = [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE].x, landmarks[mp_pose.PoseLandmark.LEFT_ANKLE].y]
        right_hip = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP].x, landmarks[mp_pose.PoseLandmark.RIGHT_HIP].y]
        right_knee = [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE].y]
        right_ankle = [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE].x, landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE].y]

        # 각도 계산
        left_angle = calculateAngle(left_hip, left_knee, left_ankle)
        right_angle = calculateAngle(right_hip, right_knee, right_ankle)

        # 화면에 각도 표시
        cv2.putText(frame, f'Left Angle: {left_angle:.2f}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(frame, f'Right Angle: {right_angle:.2f}', (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow('Exercise Analysis', frame)

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

## 5. Train Custom Model Using Scikit Learn

### 5_1. Read in Collected Data and Process

import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('coords.csv')

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df[df['class'] == 'correct_bench']

In [ ]:
X = df.drop('class', axis=1)
y = df['class']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
y_test

### 5_2. Train Machine Learning Classification Model

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [ ]:
pipelines = {
    'lr':make_pipeline(StandardScaler(), LogisticRegression()),
    'rc':make_pipeline(StandardScaler(), RidgeClassifier()),
    'rf':make_pipeline(StandardScaler(), RandomForestClassifier()),
    'gb':make_pipeline(StandardScaler(), GradientBoostingClassifier()),
}

In [ ]:
fit_models = {}
for algorithm, pipeline in pipelines.items():
    model = pipeline.fit(X_train, y_train)
    fit_models[algorithm] = model

In [ ]:
fit_models

In [ ]:
fit_models['rc'].predict(X_test)

### 5_3. Evaluate and Serialize Model

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

In [ ]:
for algorithm, model in fit_models.items():
    yhat = model.predict(X_test)
    print(algorithm, accuracy_score(y_test.values, yhat),
          precision_score(y_test.values, yhat, average="binary", pos_label="correct_bench"),
          recall_score(y_test.values, yhat, average="binary", pos_label="correct_bench"))

In [ ]:
yhat = fit_models['rf'].predict(X_test)

In [ ]:
yhat[:]

In [ ]:
with open('benchpress.pkl', 'wb') as f:
    pickle.dump(fit_models['rf'], f)

### 5_4. Make Detections with Model

In [ ]:
with open('benchpress.pkl', 'rb') as f:
    model = pickle.load(f)

In [ ]:
cap = cv2.VideoCapture(0)
counter = 0
current_stage = ''
